In [1]:
!pip uninstall -y transformers peft accelerate trl bitsandbytes datasets huggingface-hub safetensors
!pip install -q --no-cache-dir \
  transformers==4.56.1 \
  peft==0.17.0 \
  accelerate==1.10.0 \
  trl==0.23.1 \
  bitsandbytes==0.47.0 \
  datasets==4.0.0 \
  huggingface-hub==0.34.4 \
  safetensors==0.6.2 \
  sentencepiece \
  pandas==2.2.2 \
  numpy==2.0.2

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: huggingface_hub 1.6.0
Uninstalling huggingface_hub-1.6.0:
  Successfully uninstalled huggingface_hub-1.6.0
Found existing installation: safetensors 0.7.0
Uninstalling safetensors-0.7.0:
  Successfully uninstalled safetensors-0.7.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 194.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 542.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
#Imports
import re
import json
import random
import numpy as np
import pandas as pd
import torch
import sqlglot

from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

In [3]:
import transformers, accelerate, peft, trl, datasets, bitsandbytes

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("datasets:", datasets.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

torch: 2.10.0+cu128
transformers: 4.56.1
accelerate: 1.10.0
peft: 0.17.0
trl: 0.23.1
datasets: 4.0.0
bitsandbytes: 0.47.0
cuda available: True
device: NVIDIA H100 80GB HBM3


In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
DATASET_NAME = "b-mc2/sql-create-context"

MAX_SAMPLES = 10000
TRAIN_SIZE = 8000
VAL_SIZE = 1000
TEST_SIZE = 1000

MAX_LENGTH = 1024
BASELINE_EVAL_N = 100
FINAL_EVAL_N = 100

OUTPUT_DIR = "phi3_text2sql_qlora_v1"

In [5]:
#Dataset loading
dataset = load_dataset(DATASET_NAME)
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 78577
    })
})

In [6]:
#Viewing dataset

print(dataset["train"][0])
print(dataset["train"].column_names)

{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}
['answer', 'question', 'context']


In [7]:
#Train Test Eval Split

full_ds = dataset["train"].shuffle(seed=SEED).select(range(MAX_SAMPLES))

train_ds = full_ds.select(range(TRAIN_SIZE))
val_ds = full_ds.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))
test_ds = full_ds.select(range(TRAIN_SIZE + VAL_SIZE, TRAIN_SIZE + VAL_SIZE + TEST_SIZE))

print("train:", len(train_ds))
print("val:", len(val_ds))
print("test:", len(test_ds))

train: 8000
val: 1000
test: 1000


In [8]:
#Tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

pad_token: <|endoftext|>
eos_token: <|endoftext|>


In [9]:
#prompt defining

def build_prompt(question: str, context: str, answer: str | None = None) -> str:
    prompt = f"""You are a SQL generation assistant.
Given the database schema and the user question, generate a valid SQL query.
Return SQL only. Do not explain anything.

Schema:
{context}

Question:
{question}

SQL:
"""
    if answer is not None:
        prompt += answer + tokenizer.eos_token
    return prompt

In [10]:
def format_example(example):
    return {
        "text": build_prompt(
            question=example["question"],
            context=example["context"],
            answer=example["answer"],
        )
    }

train_sft = train_ds.map(format_example, remove_columns=train_ds.column_names)
val_sft = val_ds.map(format_example, remove_columns=val_ds.column_names)
test_sft = test_ds.map(format_example, remove_columns=test_ds.column_names)

print(train_sft[0]["text"][:1200])

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

You are a SQL generation assistant.
Given the database schema and the user question, generate a valid SQL query.
Return SQL only. Do not explain anything.

Schema:
CREATE TABLE table_name_50 (venue VARCHAR, away_team VARCHAR)

Question:
When Essendon played away; where did they play?

SQL:
SELECT venue FROM table_name_50 WHERE away_team = "essendon"<|endoftext|>


In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [12]:
#Loading model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)
print(f"Model memory footprint: {model.get_memory_footprint()/1e9:.2f} GB")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model memory footprint: 2.60 GB


In [13]:
#Inference Helper Function

# This helper runs the model on one question + schema pair and returns generated SQL.
def generate_sql(model, tokenizer, question, context, max_new_tokens=128):
    # Build the inference prompt without the gold answer
    prompt = build_prompt(question=question, context=context, answer=None)

    # Tokenize and truncate if necessary
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)

    # Move tokenized tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Turn off gradients because this is inference, not training
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                   # Greedy decoding for deterministic output
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Convert output tokens back into text
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove the prompt from the front and keep only the model's answer
    if decoded.startswith(prompt):
        return decoded[len(prompt):].strip()

    return decoded.strip()

In [14]:
#Testing base line example before going straight to training

# Take one validation sample and see what the base model does before fine-tuning
sample = val_ds[0]

base_pred = generate_sql(
    model=model,
    tokenizer=tokenizer,
    question=sample["question"],
    context=sample["context"],
)

# Print question, schema, gold answer, and base-model prediction
print("QUESTION:\n", sample["question"])
print("\nSCHEMA:\n", sample["context"])
print("\nGOLD SQL:\n", sample["answer"])
print("\nBASE PRED:\n", base_pred)

QUESTION:
 What is the Year of Rio?

SCHEMA:
 CREATE TABLE table_name_14 (year VARCHAR, title VARCHAR)

GOLD SQL:
 SELECT year FROM table_name_14 WHERE title = "rio"

BASE PRED:
 SELECT year FROM table_name_14 WHERE title = 'Rio';


In [15]:
#Saving baseline outputs on a small test slice

# Collect base-model predictions on a small set of test examples.
# We store these so we can compare them later with the fine-tuned model.
baseline_rows = []

for i in range(BASELINE_EVAL_N):
    ex = test_ds[i]
    pred = generate_sql(
        model=model,
        tokenizer=tokenizer,
        question=ex["question"],
        context=ex["context"],
    )
    baseline_rows.append(
        {
            "idx": i,
            "question": ex["question"],
            "context": ex["context"],
            "gold": ex["answer"],
            "base_pred": pred,
        }
    )

# Put the baseline results into a dataframe for easier inspection
baseline_df = pd.DataFrame(baseline_rows)
baseline_df.head()

,idx,question,context,gold,base_pred
0,0,Which player had a To par of 13?,"CREATE TABLE table_name_78 (player VARCHAR, to...",SELECT player FROM table_name_78 WHERE to_par ...,SELECT player FROM table_name_78 WHERE to_par ...
1,1,Who wrote season 23?,CREATE TABLE table_2409041_3 (written_by VARCH...,SELECT written_by FROM table_2409041_3 WHERE n...,SELECT written_by FROM table_2409041_3 WHERE n...
2,2,"What is Lenth Feet, when Mi From Kingston is g...",CREATE TABLE table_name_28 (length_feet VARCHA...,SELECT length_feet FROM table_name_28 WHERE mi...,SELECT length_feet FROM table_name_28 WHERE mi...
3,3,Tell me the opponents for score of 6-1 2-6 7-10,CREATE TABLE table_name_90 (opponents_in_the_f...,SELECT opponents_in_the_final FROM table_name_...,SELECT opponents_in_the_final FROM table_name_...
4,4,What is the match report from the game played ...,CREATE TABLE table_name_4 (match_report VARCHA...,SELECT match_report FROM table_name_4 WHERE da...,SELECT match_report FROM table_name_4 WHERE da...


In [16]:
# Define LoRA adapter settings.
# Only these lightweight adapter weights will be trained;
# the original base model weights remain frozen.
peft_config = LoraConfig(
    r=16,                              # Rank of the LoRA adapters
    lora_alpha=32,                     # Scaling factor
    lora_dropout=0.05,                 # Dropout for LoRA layers
    bias="none",                       # Do not train bias terms
    task_type="CAUSAL_LM",             # Causal language modeling task
    target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],  # Phi-specific target modules
)

In [17]:
# SFTConfig controls the training loop behavior.
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,     # Simulates a larger effective batch size
    num_train_epochs=1,                # Start with 1 epoch for a first real run
    max_length=MAX_LENGTH,
    gradient_checkpointing=True,       # Saves memory during training
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    bf16=False,
    logging_steps=20,                  # Print logs every 20 steps
    eval_strategy="steps",             # Run validation during training
    eval_steps=100,
    save_strategy="steps",             # Save checkpoints during training
    save_steps=100,
    save_total_limit=2,                # Keep only the latest/best few checkpoints
    load_best_model_at_end=True,       # Load the best checkpoint based on eval loss
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",                  # Disable wandb/etc for now
    packing=False,
    dataset_text_field="text",         # The field containing our training prompt+answer
    seed=SEED,
    optim="paged_adamw_8bit",          # Memory-efficient optimizer
)

In [18]:
#Trainer

# This combines the model, dataset, tokenizer, LoRA config, and training settings.
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    processing_class=tokenizer,
    peft_config=peft_config,
)

# Print how many parameters are trainable vs frozen
trainer.model.print_trainable_parameters()

Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543


In [19]:
# Launch the QLoRA fine-tuning process.
train_result = trainer.train()
train_result

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.632300,0.610355,0.659245,93207.000000,0.853966
200,0.585700,0.567756,0.578774,187008.000000,0.861967
300,0.553900,0.552293,0.589184,280710.000000,0.864716
400,0.524700,0.536271,0.544560,374291.000000,0.867077
500,0.515800,0.527855,0.557641,467495.000000,0.868397
600,0.510500,0.520773,0.545333,560047.000000,0.870076
700,0.519500,0.514430,0.547469,654245.000000,0.871183
800,0.508400,0.510214,0.527466,748268.000000,0.872129
900,0.479600,0.506984,0.526782,841492.000000,0.872418
1000,0.493400,0.505680,0.529523,935253.000000,0.872851


TrainOutput(global_step=1000, training_loss=0.5540139760971069, metrics={'train_runtime': 1623.5103, 'train_samples_per_second': 4.928, 'train_steps_per_second': 0.616, 'total_flos': 2.294168467850035e+16, 'train_loss': 0.5540139760971069, 'epoch': 1.0})

In [20]:
# Save the trained adapter weights and tokenizer locally
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('phi3_text2sql_qlora_v1/tokenizer_config.json',
 'phi3_text2sql_qlora_v1/special_tokens_map.json',
 'phi3_text2sql_qlora_v1/chat_template.jinja',
 'phi3_text2sql_qlora_v1/tokenizer.model',
 'phi3_text2sql_qlora_v1/added_tokens.json',
 'phi3_text2sql_qlora_v1/tokenizer.json')

In [21]:
# Generate SQL on the same sample used earlier, but now with the fine-tuned model
ft_pred = generate_sql(
    model=trainer.model,
    tokenizer=tokenizer,
    question=sample["question"],
    context=sample["context"],
)

print("QUESTION:\n", sample["question"])
print("\nGOLD SQL:\n", sample["answer"])
print("\nFINE-TUNED PRED:\n", ft_pred)

QUESTION:
 What is the Year of Rio?

GOLD SQL:
 SELECT year FROM table_name_14 WHERE title = "rio"

FINE-TUNED PRED:
 SELECT year FROM table_name_14 WHERE title = "rio"


In [23]:
#Defining Evaluation helpers
# Normalize SQL text so exact-match comparison is less sensitive to formatting differences
def normalize_sql(sql: str) -> str:
    sql = sql.strip()
    sql = re.sub(r"\s+", " ", sql)
    sql = sql.replace(" ;", ";")
    return sql.lower()

# Exact string match after normalization
def exact_match(pred: str, gold: str) -> bool:
    return normalize_sql(pred) == normalize_sql(gold)

# Check whether the generated SQL is syntactically parseable
def parses_as_sql(sql: str) -> bool:
    try:
        sqlglot.parse_one(sql)
        return True
    except Exception:
        return False

In [24]:
#Evaluation on test data samples

# compute simple evaluation fields for each example.
eval_rows = []

for i in range(FINAL_EVAL_N):
    ex = test_ds[i]
    pred = generate_sql(
        model=trainer.model,
        tokenizer=tokenizer,
        question=ex["question"],
        context=ex["context"],
    )

    eval_rows.append(
        {
            "idx": i,
            "question": ex["question"],
            "gold": ex["answer"],
            "ft_pred": pred,
            "ft_exact_match": exact_match(pred, ex["answer"]),
            "ft_parses": parses_as_sql(pred),
        }
    )

# Store the results in a dataframe
ft_eval_df = pd.DataFrame(eval_rows)
ft_eval_df.head()

,idx,question,gold,ft_pred,ft_exact_match,ft_parses
0,0,Which player had a To par of 13?,SELECT player FROM table_name_78 WHERE to_par ...,SELECT player FROM table_name_78 WHERE to_par ...,False,True
1,1,Who wrote season 23?,SELECT written_by FROM table_2409041_3 WHERE n...,SELECT written_by FROM table_2409041_3 WHERE n...,True,True
2,2,"What is Lenth Feet, when Mi From Kingston is g...",SELECT length_feet FROM table_name_28 WHERE mi...,SELECT length_feet FROM table_name_28 WHERE mi...,True,True
3,3,Tell me the opponents for score of 6-1 2-6 7-10,SELECT opponents_in_the_final FROM table_name_...,SELECT opponents_in_the_final FROM table_name_...,True,True
4,4,What is the match report from the game played ...,SELECT match_report FROM table_name_4 WHERE da...,SELECT match_report FROM table_name_4 WHERE da...,True,True


In [25]:
#Printing Aggregate Fint tuned metrics

print("Fine-tuned exact match:", ft_eval_df["ft_exact_match"].mean())
print("Fine-tuned parse rate:", ft_eval_df["ft_parses"].mean())

Fine-tuned exact match: 0.82
Fine-tuned parse rate: 1.0


In [26]:
#base vs fine tuned performance
# Merge baseline and fine-tuned outputs so we can compare them side by side
comparison_df = baseline_df.merge(ft_eval_df, on=["idx", "question", "gold"], how="inner")

# Compute exact match and parseability for the base model too
comparison_df["base_exact_match"] = comparison_df.apply(
    lambda row: exact_match(row["base_pred"], row["gold"]), axis=1
)
comparison_df["base_parses"] = comparison_df["base_pred"].apply(parses_as_sql)

# Print summary metrics
print("Base exact match:", comparison_df["base_exact_match"].mean())
print("Base parse rate:", comparison_df["base_parses"].mean())
print("FT exact match:", comparison_df["ft_exact_match"].mean())
print("FT parse rate:", comparison_df["ft_parses"].mean())

Base exact match: 0.0
Base parse rate: 0.92
FT exact match: 0.82
FT parse rate: 1.0


In [27]:
#Example comparisons

# gold answer vs base prediction vs fine-tuned prediction
comparison_df[[
    "question",
    "gold",
    "base_pred",
    "ft_pred",
    "base_exact_match",
    "ft_exact_match",
    "base_parses",
    "ft_parses",
]].head(10)

,question,gold,base_pred,ft_pred,base_exact_match,ft_exact_match,base_parses,ft_parses
0,Which player had a To par of 13?,SELECT player FROM table_name_78 WHERE to_par ...,SELECT player FROM table_name_78 WHERE to_par ...,SELECT player FROM table_name_78 WHERE to_par ...,False,False,True,True
1,Who wrote season 23?,SELECT written_by FROM table_2409041_3 WHERE n...,SELECT written_by FROM table_2409041_3 WHERE n...,SELECT written_by FROM table_2409041_3 WHERE n...,False,True,True,True
2,"What is Lenth Feet, when Mi From Kingston is g...",SELECT length_feet FROM table_name_28 WHERE mi...,SELECT length_feet FROM table_name_28 WHERE mi...,SELECT length_feet FROM table_name_28 WHERE mi...,False,True,True,True
3,Tell me the opponents for score of 6-1 2-6 7-10,SELECT opponents_in_the_final FROM table_name_...,SELECT opponents_in_the_final FROM table_name_...,SELECT opponents_in_the_final FROM table_name_...,False,True,True,True
4,What is the match report from the game played ...,SELECT match_report FROM table_name_4 WHERE da...,SELECT match_report FROM table_name_4 WHERE da...,SELECT match_report FROM table_name_4 WHERE da...,False,True,True,True
5,"What is Pronunciation Spelled Free, when Pronu...",SELECT pronunciation_spelled_free FROM table_n...,SELECT pronunciation_spelled_free FROM table_n...,SELECT pronunciation_spelled_free FROM table_n...,False,True,True,True
6,What ground had Portsmouth reserves as an oppo...,SELECT ground FROM table_name_98 WHERE opponen...,SELECT ground FROM table_name_98 WHERE opponen...,SELECT ground FROM table_name_98 WHERE opponen...,False,True,True,True
7,Tell me the total receipts for tom tancredo,SELECT total_receipts FROM table_name_44 WHERE...,SELECT total_receipts FROM table_name_44 WHERE...,SELECT total_receipts FROM table_name_44 WHERE...,False,True,True,True
8,Which Group position has Result F–A of 0–1 on ...,SELECT group_position FROM table_name_50 WHERE...,SELECT group_position FROM table_name_50 WHERE...,SELECT group_position FROM table_name_50 WHERE...,False,True,True,True
9,When was the record 27-25?,SELECT date FROM table_name_64 WHERE record = ...,SELECT date FROM table_name_64 WHERE record = ...,SELECT date FROM table_name_64 WHERE record = ...,False,True,True,True


In [29]:
#proper functioning verification

row = comparison_df.iloc[4]

print("QUESTION:\n", row["question"])
print("\nGOLD:\n", row["gold"])
print("\nBASE PRED:\n", row["base_pred"])
print("\nFT PRED:\n", row["ft_pred"])

print("\nBASE EXACT:", row["base_exact_match"])
print("FT EXACT:", row["ft_exact_match"])
print("BASE PARSES:", row["base_parses"])
print("FT PARSES:", row["ft_parses"])

QUESTION:
 What is the match report from the game played on 25 april 2009?

GOLD:
 SELECT match_report FROM table_name_4 WHERE date = "25 april 2009"

BASE PRED:
 SELECT match_report FROM table_name_4 WHERE date = '2009-04-25';

FT PRED:
 SELECT match_report FROM table_name_4 WHERE date = "25 april 2009"

BASE EXACT: False
FT EXACT: True
BASE PARSES: True
FT PARSES: True


In [30]:
#Label failure types for error analysis

# Rough error categorization function.
def label_error(pred: str, gold: str) -> str:
    if not parses_as_sql(pred):
        return "invalid_sql"
    if exact_match(pred, gold):
        return "exact_match"

    pred_norm = normalize_sql(pred)
    gold_norm = normalize_sql(gold)

    # If the gold SQL has a JOIN but the prediction does not
    if " join " in gold_norm and " join " not in pred_norm:
        return "missing_join"

    # If the gold SQL has a WHERE clause but the prediction does not
    if " where " in gold_norm and " where " not in pred_norm:
        return "missing_where"

    # If the gold SQL uses aggregation but the prediction does not
    if any(func in gold_norm for func in ["count(", "avg(", "sum(", "min(", "max("]) and not any(
        func in pred_norm for func in ["count(", "avg(", "sum(", "min(", "max("]
    ):
        return "aggregation_error"

    return "other_mismatch"

# error labeling to both base and fine-tuned predictions
comparison_df["base_error_type"] = comparison_df.apply(
    lambda row: label_error(row["base_pred"], row["gold"]), axis=1
)
comparison_df["ft_error_type"] = comparison_df.apply(
    lambda row: label_error(row["ft_pred"], row["gold"]), axis=1
)

# Print error breakdowns
print("Base error breakdown:")
print(comparison_df["base_error_type"].value_counts(dropna=False))

print("\nFT error breakdown:")
print(comparison_df["ft_error_type"].value_counts(dropna=False))

Base error breakdown:
base_error_type
other_mismatch       84
invalid_sql           8
aggregation_error     6
missing_where         1
missing_join          1
Name: count, dtype: int64

FT error breakdown:
ft_error_type
exact_match          82
other_mismatch       17
aggregation_error     1
Name: count, dtype: int64


In [31]:
#Saving evaluation outputs

comparison_df.to_csv("phi3_text2sql_comparison.csv", index=False)
ft_eval_df.to_csv("phi3_text2sql_ft_eval.csv", index=False)

print("Saved:")
print("- phi3_text2sql_comparison.csv")
print("- phi3_text2sql_ft_eval.csv")
print(f"- adapter/tokenizer folder: {OUTPUT_DIR}")

Saved:
- phi3_text2sql_comparison.csv
- phi3_text2sql_ft_eval.csv
- adapter/tokenizer folder: phi3_text2sql_qlora_v1


In [32]:
#Experiment Summary

#JSON-style summary of the experiment results
summary = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "train_size": TRAIN_SIZE,
    "val_size": VAL_SIZE,
    "test_size": TEST_SIZE,
    "baseline_eval_n": BASELINE_EVAL_N,
    "final_eval_n": FINAL_EVAL_N,
    "base_exact_match": float(comparison_df["base_exact_match"].mean()),
    "base_parse_rate": float(comparison_df["base_parses"].mean()),
    "ft_exact_match": float(comparison_df["ft_exact_match"].mean()),
    "ft_parse_rate": float(comparison_df["ft_parses"].mean()),
}

print(json.dumps(summary, indent=2))

{
  "model": "microsoft/Phi-3-mini-4k-instruct",
  "dataset": "b-mc2/sql-create-context",
  "train_size": 8000,
  "val_size": 1000,
  "test_size": 1000,
  "baseline_eval_n": 100,
  "final_eval_n": 100,
  "base_exact_match": 0.0,
  "base_parse_rate": 0.92,
  "ft_exact_match": 0.82,
  "ft_parse_rate": 1.0
}


In [33]:
#Exact improvement from fine tuning

best_examples = comparison_df[
    (comparison_df["ft_exact_match"] == True) & (comparison_df["base_exact_match"] == False)
][["question", "gold", "base_pred", "ft_pred"]]

best_examples.head(10)

,question,gold,base_pred,ft_pred
1,Who wrote season 23?,SELECT written_by FROM table_2409041_3 WHERE n...,SELECT written_by FROM table_2409041_3 WHERE n...,SELECT written_by FROM table_2409041_3 WHERE n...
2,"What is Lenth Feet, when Mi From Kingston is g...",SELECT length_feet FROM table_name_28 WHERE mi...,SELECT length_feet FROM table_name_28 WHERE mi...,SELECT length_feet FROM table_name_28 WHERE mi...
3,Tell me the opponents for score of 6-1 2-6 7-10,SELECT opponents_in_the_final FROM table_name_...,SELECT opponents_in_the_final FROM table_name_...,SELECT opponents_in_the_final FROM table_name_...
4,What is the match report from the game played ...,SELECT match_report FROM table_name_4 WHERE da...,SELECT match_report FROM table_name_4 WHERE da...,SELECT match_report FROM table_name_4 WHERE da...
5,"What is Pronunciation Spelled Free, when Pronu...",SELECT pronunciation_spelled_free FROM table_n...,SELECT pronunciation_spelled_free FROM table_n...,SELECT pronunciation_spelled_free FROM table_n...
6,What ground had Portsmouth reserves as an oppo...,SELECT ground FROM table_name_98 WHERE opponen...,SELECT ground FROM table_name_98 WHERE opponen...,SELECT ground FROM table_name_98 WHERE opponen...
7,Tell me the total receipts for tom tancredo,SELECT total_receipts FROM table_name_44 WHERE...,SELECT total_receipts FROM table_name_44 WHERE...,SELECT total_receipts FROM table_name_44 WHERE...
8,Which Group position has Result F–A of 0–1 on ...,SELECT group_position FROM table_name_50 WHERE...,SELECT group_position FROM table_name_50 WHERE...,SELECT group_position FROM table_name_50 WHERE...
9,When was the record 27-25?,SELECT date FROM table_name_64 WHERE record = ...,SELECT date FROM table_name_64 WHERE record = ...,SELECT date FROM table_name_64 WHERE record = ...
10,Who is week 3 if week 2 is Nikki Fiction?,SELECT week_3 FROM table_name_64 WHERE week_2 ...,SELECT week_3 FROM table_name_64 WHERE week_2 ...,SELECT week_3 FROM table_name_64 WHERE week_2 ...


In [34]:
#Remaining failures after fine tunining

# Extract examples where the fine-tuned model still failed.
worst_examples = comparison_df[
    (comparison_df["ft_exact_match"] == False)
][["question", "gold", "base_pred", "ft_pred", "ft_error_type"]]

worst_examples.head(10)

,question,gold,base_pred,ft_pred,ft_error_type
0,Which player had a To par of 13?,SELECT player FROM table_name_78 WHERE to_par ...,SELECT player FROM table_name_78 WHERE to_par ...,SELECT player FROM table_name_78 WHERE to_par ...,other_mismatch
15,Who is the opponent in the final with a clay s...,SELECT opponent_in_the_final FROM table_name_2...,SELECT opponent_in_the_final FROM table_name_2...,SELECT opponent_in_the_final FROM table_name_2...,other_mismatch
25,How many wickets were there when average was 2...,SELECT wickets FROM table_16570286_4 WHERE ave...,SELECT COUNT(*) FROM table_16570286_4 WHERE av...,SELECT COUNT(wickets) FROM table_16570286_4 WH...,other_mismatch
37,What is the lowest Top-25 for the open champio...,SELECT MIN(top_25) FROM table_name_55 WHERE to...,SELECT MIN(top_25) AS lowest_top_25\nFROM tabl...,SELECT MIN(top_25) FROM table_name_55 WHERE to...,other_mismatch
39,What is the total number of ends after 2006 wi...,SELECT COUNT(ends) FROM table_name_39 WHERE si...,SELECT COUNT(*) FROM table_name_39 WHERE ends ...,SELECT COUNT(ends) FROM table_name_39 WHERE si...,other_mismatch
46,What type was issued in 1964?,SELECT type FROM table_name_1 WHERE issued = 1964,SELECT type FROM table_name_1 WHERE issued = '...,"SELECT type FROM table_name_1 WHERE issued = ""...",other_mismatch
51,How many rounds did Brett go for the Strikefor...,SELECT round FROM table_name_84 WHERE event = ...,SELECT COUNT(*) FROM table_name_84 WHERE event...,SELECT COUNT(round) FROM table_name_84 WHERE e...,other_mismatch
65,Name the candidates in 1964,SELECT candidates FROM table_1341586_43 WHERE ...,SELECT candidates FROM table_1341586_43 WHERE ...,SELECT candidates FROM table_1341586_43 WHERE ...,other_mismatch
74,Which date has a Race Leader of hermann buse (...,SELECT date FROM table_name_83 WHERE race_lead...,SELECT date FROM table_name_83 WHERE race_lead...,SELECT date FROM table_name_83 WHERE race_lead...,other_mismatch
77,What is the place when less than 1 point is sc...,SELECT AVG(place) FROM table_name_77 WHERE poi...,SELECT place FROM table_name_77 WHERE points < 1;,SELECT MIN(place) FROM table_name_77 WHERE poi...,other_mismatch


In [35]:
#VERSION 2 updates

# Show full SQL strings instead of truncated previews
pd.set_option("display.max_colwidth", None)

# sqlglot expression helpers for AST analysis
from sqlglot import exp

In [36]:
#Better Normalization for strict matching

# Improved normalization:
# - lowercases
# - collapses whitespace
# - removes trailing semicolons
# - converts quoted integers like '13' or "13" into 13
# This still remains a STRING-LEVEL metric, not true semantic equivalence.
def normalize_sql_v2(sql: str) -> str:
    if sql is None:
        return ""

    sql = sql.strip().lower()
    sql = re.sub(r"\s+", " ", sql)
    sql = sql.rstrip(";")

    # Normalize quoted integers: '13' -> 13, "13" -> 13
    sql = re.sub(r"""['"](\d+)['"]""", r"\1", sql)

    return sql


def strict_match_v2(pred: str, gold: str) -> bool:
    return normalize_sql_v2(pred) == normalize_sql_v2(gold)

In [37]:
#AST based comparison

# Parse SQL into an AST and render it in a more canonical way.

def canonicalize_sql(sql: str):
    try:
        parsed = sqlglot.parse_one(sql)
        return parsed.sql(pretty=False)
    except Exception:
        return None


def canonical_match(pred: str, gold: str) -> bool:
    pred_canon = canonicalize_sql(pred)
    gold_canon = canonicalize_sql(gold)

    if pred_canon is None or gold_canon is None:
        return False

    return normalize_sql_v2(pred_canon) == normalize_sql_v2(gold_canon)

In [40]:
#Helper functions for structural analysis

from sqlglot import exp

def parse_sql(sql: str):
    try:
        return sqlglot.parse_one(sql)
    except Exception:
        return None


def extract_tables(sql: str):
    tree = parse_sql(sql)
    if tree is None:
        return set()
    return {t.name.lower() for t in tree.find_all(exp.Table) if t.name}


def extract_columns(sql: str):
    tree = parse_sql(sql)
    if tree is None:
        return set()

    cols = set()
    for c in tree.find_all(exp.Column):
        if c.name:
            cols.add(c.name.lower())
    return cols


def extract_aggregations(sql: str):
    tree = parse_sql(sql)
    if tree is None:
        return set()

    aggs = set()
    for node in tree.walk():
        cls_name = node.__class__.__name__.lower()
        if cls_name in {"count", "sum", "avg", "min", "max"}:
            aggs.add(cls_name)
    return aggs


def has_where(sql: str):
    tree = parse_sql(sql)
    if tree is None:
        return False
    return tree.find(exp.Where) is not None


def has_join(sql: str):
    tree = parse_sql(sql)
    if tree is None:
        return False
    return any(True for _ in tree.find_all(exp.Join))


def label_error_v2(pred: str, gold: str) -> str:
    pred_tree = parse_sql(pred)
    gold_tree = parse_sql(gold)

    if pred_tree is None:
        return "invalid_sql"

    if strict_match_v2(pred, gold):
        return "strict_match"

    if canonical_match(pred, gold):
        return "canonical_match"

    pred_tables = extract_tables(pred)
    gold_tables = extract_tables(gold)

    pred_cols = extract_columns(pred)
    gold_cols = extract_columns(gold)

    pred_aggs = extract_aggregations(pred)
    gold_aggs = extract_aggregations(gold)

    if gold_tables != pred_tables:
        return "table_mismatch"

    if gold_cols != pred_cols:
        return "column_mismatch"

    if has_join(gold) and not has_join(pred):
        return "missing_join"

    if has_where(gold) and not has_where(pred):
        return "missing_where"

    if gold_aggs != pred_aggs:
        return "aggregation_mismatch"

    return "literal_or_condition_mismatch"

In [41]:
#Re-Scoring comprison dataframe

# If comparison_df already exists from your earlier cells, this upgrades it.
# If not, it rebuilds it from baseline_df and ft_eval_df.

if "comparison_df" not in globals():
    comparison_df = baseline_df.merge(ft_eval_df, on=["idx", "question", "gold"], how="inner")

# Recompute base-side parse flags from the actual SQL strings
comparison_df["base_parses_v2"] = comparison_df["base_pred"].apply(lambda x: parse_sql(x) is not None)
comparison_df["ft_parses_v2"] = comparison_df["ft_pred"].apply(lambda x: parse_sql(x) is not None)

# New strict match metric (better normalization than before)
comparison_df["base_strict_match_v2"] = comparison_df.apply(
    lambda row: strict_match_v2(row["base_pred"], row["gold"]), axis=1
)
comparison_df["ft_strict_match_v2"] = comparison_df.apply(
    lambda row: strict_match_v2(row["ft_pred"], row["gold"]), axis=1
)

# Canonical / AST-based match
comparison_df["base_canonical_match"] = comparison_df.apply(
    lambda row: canonical_match(row["base_pred"], row["gold"]), axis=1
)
comparison_df["ft_canonical_match"] = comparison_df.apply(
    lambda row: canonical_match(row["ft_pred"], row["gold"]), axis=1
)

# Error buckets
comparison_df["base_error_type_v2"] = comparison_df.apply(
    lambda row: label_error_v2(row["base_pred"], row["gold"]), axis=1
)
comparison_df["ft_error_type_v2"] = comparison_df.apply(
    lambda row: label_error_v2(row["ft_pred"], row["gold"]), axis=1
)

comparison_df.head(5)

,idx,question,context,gold,base_pred,ft_pred,ft_exact_match,ft_parses,base_exact_match,base_parses,base_error_type,ft_error_type,base_parses_v2,ft_parses_v2,base_strict_match_v2,ft_strict_match_v2,base_canonical_match,ft_canonical_match,base_error_type_v2,ft_error_type_v2
0,0,Which player had a To par of 13?,"CREATE TABLE table_name_78 (player VARCHAR, to_par VARCHAR)",SELECT player FROM table_name_78 WHERE to_par = 13,SELECT player FROM table_name_78 WHERE to_par = '13';,"SELECT player FROM table_name_78 WHERE to_par = ""13""",False,True,False,True,other_mismatch,other_mismatch,True,True,True,True,True,True,strict_match,strict_match
1,1,Who wrote season 23?,"CREATE TABLE table_2409041_3 (written_by VARCHAR, no_in_season VARCHAR)",SELECT written_by FROM table_2409041_3 WHERE no_in_season = 23,SELECT written_by FROM table_2409041_3 WHERE no_in_season = '23';,SELECT written_by FROM table_2409041_3 WHERE no_in_season = 23,True,True,False,True,other_mismatch,exact_match,True,True,True,True,True,True,strict_match,strict_match
2,2,"What is Lenth Feet, when Mi From Kingston is greater than 84.5, when Length Meters is greater than 55.5, and when Name is Unnamed?","CREATE TABLE table_name_28 (length_feet VARCHAR, name VARCHAR, mi_from_kingston VARCHAR, length_meters VARCHAR)","SELECT length_feet FROM table_name_28 WHERE mi_from_kingston > 84.5 AND length_meters > 55.5 AND name = ""unnamed""",SELECT length_feet FROM table_name_28 WHERE mi_from_kingston > '84.5' AND length_meters > '55.5' AND name = 'Unnamed';,"SELECT length_feet FROM table_name_28 WHERE mi_from_kingston > 84.5 AND length_meters > 55.5 AND name = ""unnamed""",True,True,False,True,other_mismatch,exact_match,True,True,False,True,False,True,column_mismatch,strict_match
3,3,Tell me the opponents for score of 6-1 2-6 7-10,"CREATE TABLE table_name_90 (opponents_in_the_final VARCHAR, score VARCHAR)","SELECT opponents_in_the_final FROM table_name_90 WHERE score = ""6-1 2-6 7-10""",SELECT opponents_in_the_final FROM table_name_90 WHERE score = '6-1 2-6 7-10';,"SELECT opponents_in_the_final FROM table_name_90 WHERE score = ""6-1 2-6 7-10""",True,True,False,True,other_mismatch,exact_match,True,True,False,True,False,True,column_mismatch,strict_match
4,4,What is the match report from the game played on 25 april 2009?,"CREATE TABLE table_name_4 (match_report VARCHAR, date VARCHAR)","SELECT match_report FROM table_name_4 WHERE date = ""25 april 2009""",SELECT match_report FROM table_name_4 WHERE date = '2009-04-25';,"SELECT match_report FROM table_name_4 WHERE date = ""25 april 2009""",True,True,False,True,other_mismatch,exact_match,True,True,False,True,False,True,column_mismatch,strict_match


In [42]:
#Upgraded Metrics

print("===== V2 EVALUATION SUMMARY =====")
print()
print("Base parse rate:", comparison_df["base_parses_v2"].mean())
print("FT parse rate:", comparison_df["ft_parses_v2"].mean())
print()
print("Base strict match v2:", comparison_df["base_strict_match_v2"].mean())
print("FT strict match v2:", comparison_df["ft_strict_match_v2"].mean())
print()
print("Base canonical match:", comparison_df["base_canonical_match"].mean())
print("FT canonical match:", comparison_df["ft_canonical_match"].mean())

===== V2 EVALUATION SUMMARY =====

Base parse rate: 0.92
FT parse rate: 1.0

Base strict match v2: 0.12
FT strict match v2: 0.89

Base canonical match: 0.12
FT canonical match: 0.89


In [45]:
#Error Breakdown

print("===== BASE ERROR BREAKDOWN (V2) =====")
print(comparison_df["base_error_type_v2"].value_counts(dropna=False))
print()

print("===== FT ERROR BREAKDOWN (V2) =====")
print(comparison_df["ft_error_type_v2"].value_counts(dropna=False))

===== BASE ERROR BREAKDOWN (V2) =====
base_error_type_v2
column_mismatch                  70
strict_match                     12
invalid_sql                       8
literal_or_condition_mismatch     4
aggregation_mismatch              4
table_mismatch                    1
missing_join                      1
Name: count, dtype: int64

===== FT ERROR BREAKDOWN (V2) =====
ft_error_type_v2
strict_match                     89
aggregation_mismatch              6
column_mismatch                   3
literal_or_condition_mismatch     2
Name: count, dtype: int64


In [46]:
#Strict cases failed but canonical passed

# These are very useful examples.
# They show cases where plain exact match was too harsh,
# but the generated SQL was structurally/canonically equivalent.

base_recovered = comparison_df[
    (comparison_df["base_strict_match_v2"] == False) &
    (comparison_df["base_canonical_match"] == True)
][["question", "gold", "base_pred", "base_error_type_v2"]]

ft_recovered = comparison_df[
    (comparison_df["ft_strict_match_v2"] == False) &
    (comparison_df["ft_canonical_match"] == True)
][["question", "gold", "ft_pred", "ft_error_type_v2"]]

print("Base recovered by canonical match:", len(base_recovered))
print("FT recovered by canonical match:", len(ft_recovered))

Base recovered by canonical match: 0
FT recovered by canonical match: 0


In [47]:
# Strong showcase examples:
# FT canonical match succeeds while base canonical match fails

ft_beats_base = comparison_df[
    (comparison_df["ft_canonical_match"] == True) &
    (comparison_df["base_canonical_match"] == False)
][[
    "question",
    "gold",
    "base_pred",
    "ft_pred",
    "base_error_type_v2",
    "ft_error_type_v2"
]]

print("Cases where FT beats base under canonical match:", len(ft_beats_base))
display(ft_beats_base.head(15))

Cases where FT beats base under canonical match: 77


,question,gold,base_pred,ft_pred,base_error_type_v2,ft_error_type_v2
2,"What is Lenth Feet, when Mi From Kingston is greater than 84.5, when Length Meters is greater than 55.5, and when Name is Unnamed?","SELECT length_feet FROM table_name_28 WHERE mi_from_kingston > 84.5 AND length_meters > 55.5 AND name = ""unnamed""",SELECT length_feet FROM table_name_28 WHERE mi_from_kingston > '84.5' AND length_meters > '55.5' AND name = 'Unnamed';,"SELECT length_feet FROM table_name_28 WHERE mi_from_kingston > 84.5 AND length_meters > 55.5 AND name = ""unnamed""",column_mismatch,strict_match
3,Tell me the opponents for score of 6-1 2-6 7-10,"SELECT opponents_in_the_final FROM table_name_90 WHERE score = ""6-1 2-6 7-10""",SELECT opponents_in_the_final FROM table_name_90 WHERE score = '6-1 2-6 7-10';,"SELECT opponents_in_the_final FROM table_name_90 WHERE score = ""6-1 2-6 7-10""",column_mismatch,strict_match
4,What is the match report from the game played on 25 april 2009?,"SELECT match_report FROM table_name_4 WHERE date = ""25 april 2009""",SELECT match_report FROM table_name_4 WHERE date = '2009-04-25';,"SELECT match_report FROM table_name_4 WHERE date = ""25 april 2009""",column_mismatch,strict_match
5,"What is Pronunciation Spelled Free, when Pronunciation Spelled Checked is ""ɑ""?","SELECT pronunciation_spelled_free FROM table_name_24 WHERE pronunciation_spelled_checked = ""ɑ""",SELECT pronunciation_spelled_free FROM table_name_24 WHERE pronunciation_spelled_checked = 'ɑ';,"SELECT pronunciation_spelled_free FROM table_name_24 WHERE pronunciation_spelled_checked = ""ɑ""",column_mismatch,strict_match
6,What ground had Portsmouth reserves as an opponent?,"SELECT ground FROM table_name_98 WHERE opponent = ""portsmouth reserves""",SELECT ground FROM table_name_98 WHERE opponent = 'Portsmouth reserves';,"SELECT ground FROM table_name_98 WHERE opponent = ""portsmouth reserves""",column_mismatch,strict_match
7,Tell me the total receipts for tom tancredo,"SELECT total_receipts FROM table_name_44 WHERE candidate = ""tom tancredo""",SELECT total_receipts FROM table_name_44 WHERE candidate = 'tom tancredo';,"SELECT total_receipts FROM table_name_44 WHERE candidate = ""tom tancredo""",column_mismatch,strict_match
8,Which Group position has Result F–A of 0–1 on 1 november 2006?,"SELECT group_position FROM table_name_50 WHERE result_f_a = ""0–1"" AND date = ""1 november 2006""",SELECT group_position FROM table_name_50 WHERE result_f_a = '0-1' AND date = '2006-11-01';,"SELECT group_position FROM table_name_50 WHERE result_f_a = ""0–1"" AND date = ""1 november 2006""",column_mismatch,strict_match
9,When was the record 27-25?,"SELECT date FROM table_name_64 WHERE record = ""27-25""",SELECT date FROM table_name_64 WHERE record = '27-25';,"SELECT date FROM table_name_64 WHERE record = ""27-25""",column_mismatch,strict_match
10,Who is week 3 if week 2 is Nikki Fiction?,"SELECT week_3 FROM table_name_64 WHERE week_2 = ""nikki fiction""",SELECT week_3 FROM table_name_64 WHERE week_2 = 'Nikki Fiction';,"SELECT week_3 FROM table_name_64 WHERE week_2 = ""nikki fiction""",column_mismatch,strict_match
11,What was the table position for the team whose outgoing manager was Brian Laws?,"SELECT position_in_table FROM table_26593762_3 WHERE outgoing_manager = ""Brian Laws""",SELECT position_in_table FROM table_26593762_3 WHERE outgoing_manager = 'Brian Laws';,"SELECT position_in_table FROM table_26593762_3 WHERE outgoing_manager = ""Brian Laws""",column_mismatch,strict_match


In [48]:
#Comparison result saving
comparison_df.to_csv("phi3_text2sql_comparison_v2.csv", index=False)
ft_beats_base.to_csv("phi3_text2sql_ft_beats_base_v2.csv", index=False)
base_recovered.to_csv("phi3_text2sql_base_recovered_v2.csv", index=False)
ft_recovered.to_csv("phi3_text2sql_ft_recovered_v2.csv", index=False)

print("Saved:")
print("- phi3_text2sql_comparison_v2.csv")
print("- phi3_text2sql_ft_beats_base_v2.csv")
print("- phi3_text2sql_base_recovered_v2.csv")
print("- phi3_text2sql_ft_recovered_v2.csv")

Saved:
- phi3_text2sql_comparison_v2.csv
- phi3_text2sql_ft_beats_base_v2.csv
- phi3_text2sql_base_recovered_v2.csv
- phi3_text2sql_ft_recovered_v2.csv


In [49]:
#V2 Experimentation summary

summary_v2 = {
    "base_parse_rate_v2": float(comparison_df["base_parses_v2"].mean()),
    "ft_parse_rate_v2": float(comparison_df["ft_parses_v2"].mean()),
    "base_strict_match_v2": float(comparison_df["base_strict_match_v2"].mean()),
    "ft_strict_match_v2": float(comparison_df["ft_strict_match_v2"].mean()),
    "base_canonical_match": float(comparison_df["base_canonical_match"].mean()),
    "ft_canonical_match": float(comparison_df["ft_canonical_match"].mean()),
    "ft_beats_base_canonical_count": int(len(ft_beats_base)),
    "base_recovered_by_canonical": int(len(base_recovered)),
    "ft_recovered_by_canonical": int(len(ft_recovered)),
}

print(json.dumps(summary_v2, indent=2))

{
  "base_parse_rate_v2": 0.92,
  "ft_parse_rate_v2": 1.0,
  "base_strict_match_v2": 0.12,
  "ft_strict_match_v2": 0.89,
  "base_canonical_match": 0.12,
  "ft_canonical_match": 0.89,
  "ft_beats_base_canonical_count": 77,
  "base_recovered_by_canonical": 0,
  "ft_recovered_by_canonical": 0
}


In [50]:
for r in ft_beats_base.iterrows():
    print(r[1]["question"])

What is Lenth Feet, when Mi From Kingston is greater than 84.5, when Length Meters is greater than 55.5, and when Name is Unnamed?
Tell me the opponents for score of 6-1 2-6 7-10
What is the match report from the game played on 25 april 2009?
What is Pronunciation Spelled Free, when Pronunciation Spelled Checked is "ɑ"?
What ground had Portsmouth reserves as an opponent?
Tell me the total receipts for tom tancredo
Which Group position has Result F–A of 0–1 on 1 november 2006?
When was the record 27-25?
Who is week 3 if week 2 is Nikki Fiction?
What was the table position for the team whose outgoing manager was Brian Laws?
What is the number of against when the wins were 8, and a Club of South Warrnambool, with less than 0 draws?
Name the episode for run time of 22:50
What away team played at western oval?
Name the october when november is buffy tyler
What was Melbourne's away team score?
Show the race class and number of races in each class.
Which of the Latest version was released on 

In [51]:
#Downloading csv

from google.colab import files

files.download("phi3_text2sql_comparison.csv")
files.download("phi3_text2sql_ft_eval.csv")
files.download("phi3_text2sql_comparison_v2.csv")
files.download("phi3_text2sql_ft_beats_base_v2.csv")
files.download("phi3_text2sql_base_recovered_v2.csv")
files.download("phi3_text2sql_ft_recovered_v2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>